## Classificador de Embalagens de Alimentos

### Objetivo

Treinar um modelo de visão computacional capaz de classificar embalagens de alimentos em três categorias: **arroz**, **feijão** e **outros**.

### Metodologia

1. Importar bibliotecas e definir configurações
2. Pré-processar imagens com OpenCV (extração de ROI via fundo escuro)
3. Construir dataset customizado com augmentação de dados
4. Definir modelo (MobileNetV2 com transfer learning)
5. Treinar e avaliar o modelo (acurácia, precisão, recall, F1-score, matriz de confusão)
6. Inferência em imagem individual e via webcam

### Ambiente Controlado

As imagens são capturadas em ambiente controlado: fundo escuro, iluminação fixa e câmera posicionada sobre uma rampa. Isso simplifica o pré-processamento, permitindo segmentação confiável via limiarização de Otsu.

## Passo 1 — Importação das Bibliotecas

Utilizamos:
- **PyTorch + torchvision**: framework de deep learning e modelos pré-treinados
- **OpenCV**: pré-processamento de imagem e captura de webcam
- **scikit-learn**: métricas de avaliação (precision, recall, F1, confusion matrix)
- **matplotlib**: visualização de gráficos e imagens
- **PIL / NumPy / pathlib**: manipulação de imagens, arrays e caminhos

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms

print(f"PyTorch:     {torch.__version__}")
print(f"OpenCV:      {cv2.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"CUDA:        {'disponível (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'não disponível (usando CPU)'}")
print("\nBibliotecas importadas com sucesso.")

## Passo 2 — Caminhos e Constantes

Definimos os diretórios do dataset e os hiperparâmetros do treinamento. As subpastas de `dataset/` definem as categorias automaticamente.

In [ ]:
NOTEBOOK_DIR = Path(".").resolve()
DATASET_DIR = NOTEBOOK_DIR / "dataset"
MODELS_DIR = NOTEBOOK_DIR / "models"

CATEGORIES = ["arroz", "feijao", "outros"]
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 15
LR = 1e-4
VALIDATION_SPLIT = 0.2
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

# Criar diretórios se não existirem
for cat in CATEGORIES:
    (DATASET_DIR / cat).mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device:      {DEVICE}")
print(f"Categorias:  {CATEGORIES}")
print(f"IMG_SIZE:    {IMG_SIZE}")
print(f"BATCH_SIZE:  {BATCH_SIZE}")
print(f"EPOCHS:      {EPOCHS}")
print(f"LR:          {LR}")
print(f"\nDiretório do dataset: {DATASET_DIR}")
print(f"Diretório de modelos: {MODELS_DIR}")

for cat in CATEGORIES:
    count = sum(1 for f in (DATASET_DIR / cat).iterdir() if f.suffix.lower() in SUPPORTED_EXTENSIONS)
    print(f"  {cat}: {count} imagens")

## Passo 3 — Pré-processamento com OpenCV

Aproveitamos o fundo escuro do ambiente controlado para segmentar a embalagem:

1. Converter para escala de cinza
2. Aplicar Gaussian blur para suavizar ruído
3. Limiarização de Otsu para separar objeto do fundo
4. Encontrar o maior contorno (embalagem)
5. Recortar a região de interesse (ROI) com margem

In [ ]:
def extract_roi(image_bgr, padding=10):
    """Extrai a região de interesse (embalagem) usando limiarização de Otsu.

    Retorna a imagem recortada (BGR) ou a imagem original se nenhum contorno for encontrado.
    """
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        return image_bgr

    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)

    img_h, img_w = image_bgr.shape[:2]
    x1 = max(0, x - padding)
    y1 = max(0, y - padding)
    x2 = min(img_w, x + w + padding)
    y2 = min(img_h, y + h + padding)

    return image_bgr[y1:y2, x1:x2]


def preprocess_image(image_path):
    """Carrega uma imagem, extrai ROI e retorna como PIL Image (RGB)."""
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        return None
    roi = extract_roi(img_bgr)
    roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    return Image.fromarray(roi_rgb)


def preprocess_frame(frame_bgr):
    """Extrai ROI de um frame BGR e retorna como PIL Image (RGB)."""
    roi = extract_roi(frame_bgr)
    roi_rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
    return Image.fromarray(roi_rgb)


print("Funções de pré-processamento definidas: extract_roi(), preprocess_image(), preprocess_frame()")

## Passo 4 — Dataset e Transforms

Criamos um `Dataset` customizado que aplica a extração de ROI antes das transformações do torchvision. O dataset de treino usa augmentação (flip, rotação, color jitter) enquanto o de validação usa apenas resize e normalização ImageNet.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class FoodDataset(Dataset):
    """Dataset de embalagens de alimentos com extração de ROI integrada."""

    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = preprocess_image(self.image_paths[idx])
        if img is None:
            # Fallback: imagem preta
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


print("Dataset e transforms definidos.")

## Passo 5 — Carregamento e Divisão dos Dados

Carregamos todas as imagens das subpastas do dataset, embaralhamos com seed fixo e dividimos em treino (80%) e validação (20%). Se o dataset estiver vazio, as células seguintes são desabilitadas graciosamente via flag `DATASET_READY`.

In [ ]:
all_paths = []
all_labels = []

for label_idx, cat in enumerate(CATEGORIES):
    cat_dir = DATASET_DIR / cat
    files = sorted([f for f in cat_dir.iterdir() if f.suffix.lower() in SUPPORTED_EXTENSIONS])
    all_paths.extend(files)
    all_labels.extend([label_idx] * len(files))
    print(f"  {cat}: {len(files)} imagens")

total = len(all_paths)
print(f"\nTotal de imagens: {total}")

DATASET_READY = total > 0

if DATASET_READY:
    # Embaralhar com seed fixo
    rng = np.random.RandomState(SEED)
    indices = rng.permutation(total)

    split = int(total * (1 - VALIDATION_SPLIT))
    train_idx = indices[:split]
    val_idx = indices[split:]

    train_paths = [all_paths[i] for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]
    val_paths = [all_paths[i] for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]

    train_dataset = FoodDataset(train_paths, train_labels, transform=train_transforms)
    val_dataset = FoodDataset(val_paths, val_labels, transform=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(f"Treino:     {len(train_dataset)} imagens")
    print(f"Validação:  {len(val_dataset)} imagens")
else:
    print("\nDataset vazio. Coloque imagens nas subpastas de dataset/ e execute novamente.")
    print("Estrutura esperada:")
    for cat in CATEGORIES:
        print(f"  dataset/{cat}/  ← imagens de {cat}")

## Passo 6 — Visualização de Amostras

Exibimos algumas imagens do dataset para verificar que o carregamento e a extração de ROI estão funcionando corretamente.

In [ ]:
if DATASET_READY:
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    axes = axes.flatten()

    sample_indices = np.random.RandomState(SEED).choice(len(train_dataset), min(8, len(train_dataset)), replace=False)

    for i, idx in enumerate(sample_indices):
        img_tensor, label = train_dataset[idx]
        # Desnormalizar para visualização
        img = img_tensor.clone()
        for c in range(3):
            img[c] = img[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
        img = img.permute(1, 2, 0).numpy().clip(0, 1)

        axes[i].imshow(img)
        axes[i].set_title(CATEGORIES[label], fontsize=12)
        axes[i].axis("off")

    # Esconder eixos vazios
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.suptitle("Amostras do Dataset (com augmentação de treino)", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Dataset vazio — nenhuma amostra para visualizar.")

## Passo 7 — Definição do Modelo

Utilizamos **MobileNetV2** pré-treinado no ImageNet como extrator de features. Congelamos todas as camadas convolucionais e substituímos apenas o classificador final por uma camada linear para 3 classes. Isso permite treinar rapidamente com poucas imagens (transfer learning).

In [ ]:
def create_model(num_classes=3):
    """Cria MobileNetV2 com classificador customizado para num_classes."""
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    # Congelar backbone
    for param in model.features.parameters():
        param.requires_grad = False

    # Substituir classificador
    last_channel = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(last_channel, num_classes),
    )

    model = model.to(DEVICE)
    return model


model = create_model(num_classes=len(CATEGORIES))
optimizer = optim.Adam(model.classifier.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {trainable:,} / {total_params:,} ({100*trainable/total_params:.1f}%)")
print(f"Modelo criado e movido para: {DEVICE}")

## Passo 8 — Treinamento

Loop de treinamento com checkpointing: salva o modelo apenas quando a acurácia de validação melhora. Registra loss e acurácia por época para visualização posterior.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


if DATASET_READY:
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    best_model_path = MODELS_DIR / "best_food_classifier.pth"

    print(f"Iniciando treinamento por {EPOCHS} épocas...\n")

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        improved = ""
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            improved = " ← melhor modelo salvo"

        print(f"Época {epoch:2d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}{improved}")

    print(f"\nMelhor acurácia de validação: {best_val_acc:.4f}")
    print(f"Modelo salvo em: {best_model_path}")
else:
    print("Dataset vazio — treinamento ignorado.")

## Passo 9 — Curvas de Treinamento

Visualizamos a evolução da loss e acurácia durante o treinamento para identificar overfitting ou underfitting.

In [ ]:
if DATASET_READY:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    epochs_range = range(1, len(history["train_loss"]) + 1)

    ax1.plot(epochs_range, history["train_loss"], "b-o", label="Treino")
    ax1.plot(epochs_range, history["val_loss"], "r-o", label="Validação")
    ax1.set_title("Loss por Época")
    ax1.set_xlabel("Época")
    ax1.set_ylabel("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs_range, history["train_acc"], "b-o", label="Treino")
    ax2.plot(epochs_range, history["val_acc"], "r-o", label="Validação")
    ax2.set_title("Acurácia por Época")
    ax2.set_xlabel("Época")
    ax2.set_ylabel("Acurácia")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.suptitle("Curvas de Treinamento", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Dataset vazio — nenhuma curva para exibir.")

## Passo 10 — Avaliação do Modelo

Carregamos o melhor modelo salvo e geramos o relatório de classificação (precision, recall, F1-score) e a matriz de confusão sobre o conjunto de validação.

In [ ]:
if DATASET_READY:
    # Carregar melhor modelo
    model.load_state_dict(torch.load(best_model_path, weights_only=True))
    model.eval()

    all_preds = []
    all_true = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_true.extend(labels.numpy())

    # Relatório de classificação
    print("Relatório de Classificação (Validação):\n")
    print(classification_report(all_true, all_preds, target_names=CATEGORIES, zero_division=0))

    # Matriz de confusão
    cm = confusion_matrix(all_true, all_preds)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CATEGORIES)
    disp.plot(ax=ax, cmap="Blues", values_format="d")
    ax.set_title("Matriz de Confusão — Validação")
    plt.tight_layout()
    plt.show()
else:
    print("Dataset vazio — avaliação ignorada.")

## Passo 11 — Inferência em Imagem Individual

Função para classificar uma única imagem, exibindo a predição com nível de confiança e gráfico de probabilidades por categoria.

In [ ]:
def predict_image(image_path, model=model):
    """Classifica uma imagem e exibe o resultado com gráfico de confiança."""
    model.eval()

    img_pil = preprocess_image(image_path)
    if img_pil is None:
        print(f"Erro: não foi possível carregar {image_path}")
        return None, 0.0

    img_tensor = val_transforms(img_pil).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()

    pred_idx = probs.argmax()
    pred_label = CATEGORIES[pred_idx]
    confidence = probs[pred_idx]

    # Visualização
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.imshow(img_pil)
    ax1.set_title(f"Predição: {pred_label} ({confidence:.1%})", fontsize=14)
    ax1.axis("off")

    colors = ["#4CAF50" if i == pred_idx else "#90CAF9" for i in range(len(CATEGORIES))]
    bars = ax2.barh(CATEGORIES, probs, color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Probabilidade")
    ax2.set_title("Confiança por Categoria")
    for bar, prob in zip(bars, probs):
        ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
                 f"{prob:.1%}", va="center", fontsize=11)

    plt.tight_layout()
    plt.show()

    return pred_label, confidence


# Exemplo de uso (descomente e ajuste o caminho):
# predict_image("dataset/arroz/exemplo.jpg")
print("Função predict_image() definida.")
print("Uso: predict_image('caminho/para/imagem.jpg')")

## Passo 12 — Inferência via Webcam com Contagem de Objetos

Abre a câmera do computador em tempo real. Para cada frame:

1. Extrai ROI (segmenta objetos no fundo escuro)
2. Classifica a embalagem detectada
3. Exibe o label e a confiança sobre o frame
4. **Conta objetos únicos** — quando um novo objeto aparece na cena e é classificado com confiança suficiente, o contador incrementa. Usa um cooldown temporal para evitar contagem duplicada entre frames consecutivos.

Pressione **'q'** para encerrar. Se a webcam não estiver disponível, uma mensagem informativa é exibida.

In [ ]:
import time


def detect_object_present(frame_bgr, min_area_ratio=0.02):
    """Detecta se há um objeto significativo no frame usando contornos.

    Retorna (presente: bool, bounding_box: tuple ou None).
    min_area_ratio: fração mínima da área do frame para considerar um objeto.
    """
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False, None

    largest = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(largest)
    frame_area = frame_bgr.shape[0] * frame_bgr.shape[1]

    if area / frame_area < min_area_ratio:
        return False, None

    return True, cv2.boundingRect(largest)


def classify_frame(frame_bgr, model):
    """Classifica o objeto principal no frame. Retorna (label, confiança, probabilidades)."""
    model.eval()
    img_pil = preprocess_frame(frame_bgr)
    img_tensor = val_transforms(img_pil).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.softmax(outputs, dim=1)[0].cpu().numpy()

    pred_idx = probs.argmax()
    return CATEGORIES[pred_idx], probs[pred_idx], probs


def run_webcam_inference(model=model, confidence_threshold=0.6, cooldown_seconds=2.0):
    """Inferência em tempo real via webcam com contagem de objetos únicos.

    - Detecta presença de objeto via contornos
    - Classifica quando objeto está presente
    - Conta objeto como novo quando: objeto aparece após ausência (ou após cooldown)
    - cooldown_seconds: tempo mínimo entre contagens para evitar duplicatas
    """
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Webcam não disponível.")
        print("Para testar inferência sem webcam, use a função predict_image():")
        print("  predict_image('caminho/para/imagem.jpg')")
        return

    print("Webcam aberta. Pressione 'q' para encerrar.")
    print(f"Limiar de confiança para contagem: {confidence_threshold:.0%}")
    print(f"Cooldown entre contagens: {cooldown_seconds}s")
    print()

    # Estado de contagem
    counts = {cat: 0 for cat in CATEGORIES}
    total_count = 0
    object_was_present = False
    last_count_time = 0.0
    current_label = ""
    current_confidence = 0.0

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Erro ao capturar frame.")
            break

        # Detectar presença de objeto
        obj_present, bbox = detect_object_present(frame)

        if obj_present:
            # Classificar
            label, confidence, probs = classify_frame(frame, model)
            current_label = label
            current_confidence = confidence

            # Contar objeto novo: apareceu após ausência OU após cooldown
            now = time.time()
            is_new_object = (not object_was_present) or (now - last_count_time > cooldown_seconds)

            if is_new_object and confidence >= confidence_threshold:
                counts[label] += 1
                total_count += 1
                last_count_time = now
                print(f"  [{total_count}] {label} (confiança: {confidence:.1%})")

            # Desenhar bounding box
            if bbox is not None:
                x, y, w, h = bbox
                color = (0, 255, 0) if confidence >= confidence_threshold else (0, 165, 255)
                cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

            # Label no frame
            text = f"{label}: {confidence:.1%}"
            cv2.putText(frame, text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)

            object_was_present = True
        else:
            object_was_present = False
            cv2.putText(frame, "Sem objeto detectado", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (128, 128, 128), 2)

        # Exibir contagens no canto inferior
        y_offset = frame.shape[0] - 20
        count_text = f"Total: {total_count} | " + " | ".join(f"{cat}: {counts[cat]}" for cat in CATEGORIES)
        cv2.putText(frame, count_text, (10, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        cv2.imshow("Classificador de Alimentos", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

    # Resumo final
    print(f"\n--- Resumo da Sessão ---")
    print(f"Total de objetos contados: {total_count}")
    for cat in CATEGORIES:
        print(f"  {cat}: {counts[cat]}")


print("Função run_webcam_inference() definida.")
print("Execute run_webcam_inference() para iniciar a câmera.")

In [ ]:
# Descomente a linha abaixo para iniciar a webcam:
# run_webcam_inference()

## Passo 13 — Carregar Modelo Salvo

Para usar o modelo treinado em sessões futuras sem retreinar, basta carregar o arquivo `.pth` salvo na pasta `models/`.

In [ ]:
def load_saved_model(path=None):
    """Carrega um modelo salvo para inferência sem retreinamento."""
    if path is None:
        path = MODELS_DIR / "best_food_classifier.pth"
    path = Path(path)

    if not path.exists():
        print(f"Arquivo não encontrado: {path}")
        print("Treine o modelo primeiro ou verifique o caminho.")
        return None

    loaded_model = create_model(num_classes=len(CATEGORIES))
    loaded_model.load_state_dict(torch.load(path, weights_only=True, map_location=DEVICE))
    loaded_model.eval()
    print(f"Modelo carregado de: {path}")
    return loaded_model


# Exemplo de uso:
# modelo_salvo = load_saved_model()
# predict_image("caminho/para/imagem.jpg", model=modelo_salvo)
# run_webcam_inference(model=modelo_salvo)

saved_path = MODELS_DIR / "best_food_classifier.pth"
if saved_path.exists():
    print(f"Modelo salvo encontrado: {saved_path}")
    print(f"Tamanho: {saved_path.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print("Nenhum modelo salvo encontrado. Treine o modelo primeiro.")

## Conclusão

Neste notebook implementamos um pipeline completo de classificação de embalagens de alimentos:

1. **Pré-processamento**: extração automática de ROI via limiarização de Otsu, aproveitando o fundo escuro do ambiente controlado
2. **Transfer Learning**: MobileNetV2 pré-treinado no ImageNet com apenas o classificador final retreinado — eficiente mesmo com poucos dados
3. **Augmentação de dados**: flip, rotação e color jitter para aumentar a robustez do modelo
4. **Avaliação**: métricas completas (precision, recall, F1-score) e matriz de confusão
5. **Inferência em tempo real**: classificação via webcam com contagem de objetos únicos usando cooldown temporal
6. **Persistência**: checkpointing do melhor modelo para uso futuro sem retreinamento

Para adicionar novas categorias, basta:
- Adicionar o nome à lista `CATEGORIES`
- Criar a subpasta correspondente em `dataset/`
- Colocar as imagens e retreinar